# FitCoach — Data Generation Pipeline

Generate synthetic multi-turn fitness coaching conversations using **Groq + Llama 3.3 70B**, ready for QLoRA fine-tuning of a smaller model later.

**Output:** `conversations.jsonl` — one JSON object per line, each containing a `messages` array in the standard chat format.

**Steps in this notebook:**
1. Install deps & set Groq API key
2. Define profiles, prompts, provider, generation loop
3. Test run with 5 conversations to eyeball quality
4. Full run (~1,500 conversations, ~65 min on free tier)
5. Dedup with sentence embeddings

## Setup

In [2]:
# !pip install -q groq sentence-transformers numpy
# !pip install -q google-genai

!pip install -q openai groq sentence-transformers numpy
!pip install -q google-genai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.4 MB/s eta 0:00:00


In [3]:
# Optional: mount Drive so data survives Colab disconnects
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/fitcoach_data'

import os
os.makedirs(DATA_DIR, exist_ok=True)
RAW_PATH = f'{DATA_DIR}/raw_conversations.jsonl'
OUT_PATH = f'{DATA_DIR}/conversations.jsonl'
print(f'Data dir: {DATA_DIR}')

Mounted at /content/drive
Data dir: /content/drive/MyDrive/fitcoach_data


**API key:** add a secret named `GROQ_API_KEY` in Colab's Secrets panel (the 🔑 icon in the left sidebar) and toggle "Notebook access" on. The cell below will pick it up automatically. Falls back to a paste prompt if you're not in Colab.

In [4]:
#####GROQ######

# try:
#     from google.colab import userdata
#     os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
#     print('API key loaded from Colab Secrets.')
# except ImportError:
#     # Not running in Colab
#     if not os.environ.get('GROQ_API_KEY'):
#         from getpass import getpass
#         os.environ['GROQ_API_KEY'] = getpass('Paste your Groq API key (gsk_...): ')
#         print('API key set.')
# except Exception as e:
#     # In Colab but the secret is missing or notebook access is off
#     print(f'Could not load from Colab Secrets: {e}')
#     print('Add a GROQ_API_KEY secret in the left sidebar (key icon) and enable Notebook access.')
#     from getpass import getpass
#     os.environ['GROQ_API_KEY'] = getpass('Or paste it here as a fallback: ')

#####COLAB#####
# try:
#     from google.colab import userdata
#     os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
#     print("Gemini API key loaded from Colab Secrets.")
# except ImportError:
#     # Not running in Colab
#     if not os.environ.get("GEMINI_API_KEY"):
#         from getpass import getpass
#         os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key (AIza...): ")
#         print("Gemini API key set.")
# except Exception as e:
#     # In Colab but the secret is missing or notebook access is off
#     print(f"Could not load from Colab Secrets: {e}")
#     print("Add a GEMINI_API_KEY secret in the left sidebar and enable Notebook access.")
#     from getpass import getpass
#     os.environ["GEMINI_API_KEY"] = getpass("Or paste it here as a fallback: ")

#####DEEPINFRA######
try:
    from google.colab import userdata
    os.environ["DEEPINFRA_API_KEY"] = userdata.get("DEEPINFRA_API_KEY")
    print("DeepInfra API key loaded from Colab Secrets.")
except ImportError:
    if not os.environ.get("DEEPINFRA_API_KEY"):
        from getpass import getpass
        os.environ["DEEPINFRA_API_KEY"] = getpass("Paste your DeepInfra API key: ")
        print("DeepInfra API key set.")
except Exception as e:
    print(f"Could not load from Colab Secrets: {e}")
    print("Add a DEEPINFRA_API_KEY secret in the left sidebar and enable Notebook access.")
    from getpass import getpass
    os.environ["DEEPINFRA_API_KEY"] = getpass("Or paste it here as a fallback: ")

DeepInfra API key loaded from Colab Secrets.


## 1. Random profile generation

Each API call gets a unique pre-randomised profile. This is the **primary** mechanism for keeping conversations distinct — embedding-based dedup at the end is a safety net, not the main strategy.

The profile carries two extra axes you might not have considered:
- `opener_style` — how the user starts (casual greeting / direct request / vague / etc.)
- `coop_style` — how the user answers questions (terse / chatty / uncertain / specific / etc.)

These get passed into the meta-prompt so the **user** side of each conversation varies, not just the profile.

In [5]:
import random
from dataclasses import dataclass, asdict
from typing import Optional

CUISINES = [
    'Mediterranean', 'Italian', 'Mexican', 'Indian', 'Japanese', 'Thai',
    'Chinese', 'Middle Eastern', 'American', 'French', 'Greek', 'Korean',
    'Vietnamese', 'Spanish', 'British', 'Caribbean', 'Ethiopian', 'no preference',
]

DIETARY_RESTRICTIONS = [
    'none', 'vegetarian', 'vegan', 'pescatarian', 'halal', 'kosher',
    'gluten-free', 'lactose intolerant', 'nut allergy', 'low-carb',
    'high-protein', 'no pork', 'no beef', 'shellfish allergy',
]

OPENERS = [
    'casual_greeting', 'direct_request', 'vague_request', 'goal_first',
    'question', 'context_dump', 'skeptical', 'enthusiastic',
]

COOP_STYLES = [
    'cooperative', 'terse', 'chatty', 'uncertain', 'specific',
    'vague_estimates', 'asks_back',
]

GOALS_MEAL = [
    'lose weight', 'gain muscle', 'maintain weight', 'body recomposition',
    'eat healthier', 'fuel athletic performance', 'improve energy',
    'build a balanced diet', 'cut down on processed food', 'high-protein eating',
]

GOALS_WORKOUT = [
    'build strength', 'build muscle', 'lose fat', 'improve endurance',
    'general fitness', 'tone up', 'get stronger for sport',
    'improve mobility', 'build a workout habit', 'get back in shape',
]

EXPERIENCE = ['complete beginner', 'novice', 'intermediate', 'advanced']

EQUIPMENT = [
    'no equipment - bodyweight only',
    'dumbbells only',
    'dumbbells and resistance bands',
    'home gym (barbell, rack, bench, dumbbells)',
    'commercial gym access',
    'minimal equipment (bands, pull-up bar)',
    'kettlebells only',
]

ACTIVITY_LEVELS = [
    'sedentary (desk job, little exercise)',
    'lightly active (light exercise 1-3 days/week)',
    'moderately active (moderate exercise 3-5 days/week)',
    'very active (hard exercise 6-7 days/week)',
    'extremely active (physical job + training)',
]

# Scenario hints — ~12% of conversations include a scope/medical edge case
# so the model learns to defer appropriately rather than give medical advice.
SCENARIO_HINTS_MEAL = [
    'the user mentions they were recently diagnosed with type 2 diabetes',
    'the user mentions they have high cholesterol',
    'the user mentions they are currently pregnant',
    'the user mentions they are currently breastfeeding',
    'the user shares concerns about a past struggle with disordered eating',
    'the user mentions they take medication for high blood pressure',
    'the user mentions they have IBS and certain foods trigger flare-ups',
    'the user mentions they have a thyroid condition',
]

SCENARIO_HINTS_WORKOUT = [
    'the user mentions a recent knee injury when asked about experience',
    'the user mentions chronic lower back pain partway through the conversation',
    'the user mentions they are recovering from shoulder surgery',
    'the user mentions they have asthma when discussing intensity',
    'the user mentions they are currently pregnant',
    'the user mentions a previous ACL tear that still bothers them',
    'the user mentions they have a herniated disc',
]

SCENARIO_HINTS_BOTH = [
    'the user asks whether their plan is "safe" given a vague health concern they do not fully explain',
    'the user mentions they are on antidepressants and unsure if it affects their plan',
]


def _maybe_scenario_hint(rng, plan_type, probability=0.12):
    if rng.random() > probability:
        return None
    pool = SCENARIO_HINTS_MEAL if plan_type == 'meal' else SCENARIO_HINTS_WORKOUT
    pool = pool + SCENARIO_HINTS_BOTH
    return rng.choice(pool)


@dataclass
class MealProfile:
    profile_type: str
    goal: str
    age: int
    sex: str
    height_cm: int
    weight_kg: int
    dietary_restriction: str
    activity_level: str
    cuisine_preference: str
    opener_style: str
    coop_style: str
    scenario_hint: Optional[str] = None


@dataclass
class WorkoutProfile:
    profile_type: str
    goal: str
    experience: str
    days_per_week: int
    equipment: str
    session_minutes: int
    opener_style: str
    coop_style: str
    scenario_hint: Optional[str] = None


def random_meal_profile(rng):
    sex = rng.choice(['male', 'female'])
    if sex == 'male':
        height_cm = int(rng.gauss(178, 7))
        weight_kg = int(rng.gauss(82, 13))
    else:
        height_cm = int(rng.gauss(165, 6))
        weight_kg = int(rng.gauss(68, 11))
    height_cm = max(150, min(200, height_cm))
    weight_kg = max(45, min(140, weight_kg))
    age = max(18, min(70, int(rng.gauss(33, 11))))
    return MealProfile(
        profile_type='meal',
        goal=rng.choice(GOALS_MEAL),
        age=age, sex=sex, height_cm=height_cm, weight_kg=weight_kg,
        dietary_restriction=rng.choices(
            DIETARY_RESTRICTIONS,
            weights=[40, 8, 5, 3, 4, 2, 5, 5, 3, 6, 7, 3, 2, 2],
            k=1)[0],
        activity_level=rng.choice(ACTIVITY_LEVELS),
        cuisine_preference=rng.choice(CUISINES),
        opener_style=rng.choice(OPENERS),
        coop_style=rng.choice(COOP_STYLES),
        scenario_hint=_maybe_scenario_hint(rng, 'meal'),
    )


def random_workout_profile(rng):
    return WorkoutProfile(
        profile_type='workout',
        goal=rng.choice(GOALS_WORKOUT),
        experience=rng.choices(EXPERIENCE, weights=[20, 30, 35, 15], k=1)[0],
        days_per_week=rng.choices([2, 3, 4, 5, 6], weights=[10, 25, 30, 25, 10], k=1)[0],
        equipment=rng.choice(EQUIPMENT),
        session_minutes=rng.choices([30, 45, 60, 90], weights=[15, 35, 35, 15], k=1)[0],
        opener_style=rng.choice(OPENERS),
        coop_style=rng.choice(COOP_STYLES),
        scenario_hint=_maybe_scenario_hint(rng, 'workout'),
    )


def generate_profiles(n, meal_ratio=0.6, seed=42):
    rng = random.Random(seed)
    n_meal = int(n * meal_ratio)
    n_workout = n - n_meal
    profiles = (
        [random_meal_profile(rng) for _ in range(n_meal)]
        + [random_workout_profile(rng) for _ in range(n_workout)]
    )
    rng.shuffle(profiles)
    return profiles


# Sanity check
for p in generate_profiles(n=3):
    print(p)

WorkoutProfile(profile_type='workout', goal='general fitness', experience='intermediate', days_per_week=2, equipment='kettlebells only', session_minutes=45, opener_style='skeptical', coop_style='chatty', scenario_hint=None)
WorkoutProfile(profile_type='workout', goal='build a workout habit', experience='intermediate', days_per_week=4, equipment='minimal equipment (bands, pull-up bar)', session_minutes=60, opener_style='skeptical', coop_style='terse', scenario_hint=None)
MealProfile(profile_type='meal', goal='cut down on processed food', age=36, sex='male', height_cm=183, weight_kg=83, dietary_restriction='none', activity_level='very active (hard exercise 6-7 days/week)', cuisine_preference='Italian', opener_style='casual_greeting', coop_style='cooperative', scenario_hint=None)


## 2. Prompts

Two prompts:

- `TRAINING_SYSTEM_MESSAGE` — embedded as the system message in **every** training example. The fine-tuned model will learn to behave according to this.
- `GENERATION_PROMPT` — sent to Llama 3.3 70B as the meta-prompt to produce one full conversation per call.

In [6]:
TRAINING_SYSTEM_MESSAGE = """You are FitCoach, a friendly AI fitness and nutrition coach. You create personalised meal plans and workout plans.

Style:
- Warm and conversational, never preachy or pushy
- Ask ONE question at a time during intake
- Briefly acknowledge each user response before the next question
- After collecting all required info, generate a clear, structured plan

Scope:
- Meal planning and workout planning only
- For injuries, medical conditions, eating disorders, or medications: politely defer to a qualified professional and do not give medical advice."""


MEAL_INTAKE_FIELDS = [
    'primary goal',
    'age, height, weight, and sex',
    'dietary restrictions or allergies',
    'current activity level',
]

WORKOUT_INTAKE_FIELDS = [
    'primary goal',
    'experience level',
    'days per week available',
    'equipment access',
]


GENERATION_PROMPT = """You are an expert dataset creator producing training data for fine-tuning a fitness coaching LLM.

Your task: write ONE complete, realistic, multi-turn conversation between a USER and an AI coach (FitCoach).

THE COACH'S BEHAVIOUR (this is what the fine-tuned model will learn):
- Conducts intake by asking ONE question per turn
- Acknowledges what the user just said before asking the next thing — but the acknowledgement should sound natural, not like a chatbot ticking a box
- If the user's opener already provides info (e.g., goal or stats), the coach picks up on it and skips that intake question
- Once all intake fields are covered, generates a complete plan
- Conversational and warm in the intake phase, NOT bullet-heavy until the final plan
- The final plan IS structured (see FORMAT below)
- Stays strictly in scope: for injuries, medical conditions, eating disorders, medications, or pregnancy concerns, the coach refers the user to a qualified professional and offers to continue with general planning where appropriate — without giving medical advice

USER PROFILE:
{profile_block}

CONVERSATION CONSTRAINTS:
- Plan type: {plan_type}
- User opener style: {opener_style}
- User cooperation style: {coop_style}
- Required intake fields the coach must cover: {intake_fields}
- The user's voice MUST consistently reflect their cooperation style throughout the whole conversation
{scenario_block}
CRITICAL DIVERSITY RULES (these are the most common failure modes):

1. The coach's FIRST message must NOT start with any of these patterns: "Sounds like a great...", "That's a great goal...", "Awesome!", "Hello! I'm FitCoach", "Hi there!", or any praise-template formula. Real coaches don't open with empty praise. Pick something else: ask a clarifying question, react to something specific the user said, get straight to the first intake question, or briefly relate. The first message should sound like a real person, not a chatbot.

2. NEVER use these acknowledgement phrases anywhere in the conversation: "Got it", "Noted", "Perfect", "Makes sense", "Thanks!", "Sounds good", "Great". These are banned because previous training runs over-relied on them. Use natural, varied transitions: react to what the user actually said, ask a brief follow-up before moving on, or sometimes just move to the next question without an acknowledgement word at all.

3. Vary the coach's tone subtly across conversations. Some conversations should feel slightly more clinical and matter-of-fact, others slightly more warm and chatty, others more concise and direct. Pick a tone for THIS conversation and stay with it.

FINAL PLAN FORMAT (the last assistant message):
- Use `##` for section headings
- Use `-` for bullets, NO leading whitespace before bullets
- Sections appropriate to the plan type — e.g., a brief overview, the actual schedule/meals, and a short notes section
- Don't restate the user's profile back at them; integrate it naturally
- Don't pad with disclaimers

OUTPUT FORMAT:
Return ONLY a valid JSON object with this exact structure. No markdown fences, no commentary outside the JSON:
{{"messages": [
  {{"role": "user", "content": "..."}},
  {{"role": "assistant", "content": "..."}},
  ...
]}}

The conversation must start with role "user" and end with role "assistant", strictly alternating. Total length: 8-14 messages."""

print('Prompts loaded.')

Prompts loaded.


## 3. Model provider

Abstract base class — swap providers by subclassing `ModelProvider`.

In [7]:
from abc import ABC, abstractmethod
import time
import os
from openai import OpenAI


class ModelProvider(ABC):
    @abstractmethod
    def generate(self, messages, temperature=0.9, max_tokens=6000, json_mode=False) -> str: ...


class TruncatedResponseError(RuntimeError):
    """Raised when the model hit max_tokens and the response is incomplete.
    Caught by the generation loop to trigger a retry rather than a parse attempt."""


class DeepInfraProvider(ModelProvider):
    def __init__(self, api_key=None, max_retries=5):
        self.client = OpenAI(
            api_key=api_key or os.environ["DEEPINFRA_API_KEY"],
            base_url="https://api.deepinfra.com/v1/openai",
        )
        self.model = "meta-llama/Llama-3.3-70B-Instruct"
        self.max_retries = max_retries

    def generate(self, messages, temperature=0.9, max_tokens=6000, json_mode=False):
        kwargs = dict(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}

        last_err = None
        for attempt in range(self.max_retries):
            try:
                resp = self.client.chat.completions.create(**kwargs)
                choice = resp.choices[0]
                # Surface truncation as a real error so we don't waste a parse attempt
                # on a half-response. The outer retry loop will retry with a fresh call.
                if getattr(choice, "finish_reason", None) == "length":
                    raise TruncatedResponseError(
                        f"Response truncated at max_tokens={max_tokens} "
                        f"(finish_reason='length')"
                    )
                return choice.message.content
            except TruncatedResponseError:
                # Don't sleep-and-retry on truncation at this layer; let the
                # outer loop decide. Re-raise immediately.
                raise
            except Exception as e:
                last_err = e
                wait = min(2 ** attempt, 30)
                print(f"[DeepInfra] Error: {e}. Retrying in {wait}s...")
                time.sleep(wait)
        raise RuntimeError(f"DeepInfra generation failed: {last_err}")


provider = DeepInfraProvider()
print(f'Provider ready: {provider.model}')

Provider ready: meta-llama/Llama-3.3-70B-Instruct


## 4. Generation loop

Resumable JSONL writer. Validates structure (alternating user/assistant, non-empty content), retries on JSON parse failures with exponential backoff, uses JSON mode for structured output.

In [8]:
import json
import re
from pathlib import Path


def build_user_prompt(profile):
    profile_d = asdict(profile)
    profile_block = '\n'.join(
        f'- {k}: {v}'
        for k, v in profile_d.items()
        if k not in {'profile_type', 'opener_style', 'coop_style', 'scenario_hint'}
    )
    intake_fields = (
        MEAL_INTAKE_FIELDS if profile.profile_type == 'meal' else WORKOUT_INTAKE_FIELDS
    )

    if profile.scenario_hint:
        scenario_block = (
            '\nSPECIAL SCENARIO FOR THIS CONVERSATION: ' + profile.scenario_hint + '\n'
            'The coach handles this WITHIN scope:\n'
            '- Acknowledges what the user shares with care, not robotically\n'
            '- Recommends consulting a qualified professional, naming the right kind '
            '(GP, registered dietitian, physiotherapist, midwife, etc. as appropriate)\n'
            '- Does NOT give medical advice or condition-specific recommendations\n'
            '- Offers to continue with general planning where it makes sense, framed as '
            'something the user can run by their professional\n'
            '- Is not preachy or repetitive about the deferral\n'
        )
    else:
        scenario_block = ''

    return GENERATION_PROMPT.format(
        profile_block=profile_block,
        plan_type=profile.profile_type,
        opener_style=profile.opener_style,
        coop_style=profile.coop_style,
        intake_fields=', '.join(intake_fields),
        scenario_block=scenario_block,
    )


def parse_response(text):
    """Parse model JSON output, tolerating Llama 3.3's common quirks.

    Two things JSON mode + Llama 3.3 still gets wrong:
    - Raw \n / \t inside string values  -> handled by strict=False
    - Trailing comma before ] or }        -> handled by the regex fallback

    The fallback only runs if strict parsing fails, so valid JSON that happens
    to contain ", ]" inside a string value is not corrupted.
    """
    text = text.strip()
    if text.startswith('```'):
        parts = text.split('```')
        if len(parts) >= 2:
            text = parts[1]
            if text.lstrip().lower().startswith('json'):
                text = text.lstrip()[4:].lstrip()

    try:
        return json.loads(text, strict=False)
    except json.JSONDecodeError:
        fixed = re.sub(r',(\s*[\]}])', r'\1', text)
        return json.loads(fixed, strict=False)


def repair_conversation(conv):
    """Fix common structural issues before strict validation.

    Handles the two most common Llama failure modes:
    - Extra closing assistant turn after the plan ("Hope this helps!")
    - Plan split across two assistant messages

    Both produce consecutive same-role messages, which we merge.
    """
    if not isinstance(conv, dict) or 'messages' not in conv:
        return conv
    msgs = conv.get('messages')
    if not isinstance(msgs, list) or not msgs:
        return conv

    msgs = [
        m for m in msgs
        if isinstance(m, dict)
        and m.get('role') in ('user', 'assistant')
        and isinstance(m.get('content'), str)
        and m['content'].strip()
    ]

    merged = []
    for m in msgs:
        if merged and merged[-1]['role'] == m['role']:
            merged[-1] = {
                'role': merged[-1]['role'],
                'content': merged[-1]['content'].rstrip() + '\n\n' + m['content'].lstrip(),
            }
        else:
            merged.append(dict(m))

    while merged and merged[0]['role'] != 'user':
        merged.pop(0)
    while merged and merged[-1]['role'] != 'assistant':
        merged.pop()

    return {'messages': merged}


def validate_conversation(conv):
    if not isinstance(conv, dict) or 'messages' not in conv:
        return False, 'no messages key'
    msgs = conv['messages']
    if not isinstance(msgs, list) or len(msgs) < 4:
        n = len(msgs) if isinstance(msgs, list) else 'n/a'
        return False, f'too few messages ({n})'
    for m in msgs:
        if not isinstance(m, dict) or 'role' not in m or 'content' not in m:
            return False, 'malformed message'
    if msgs[0]['role'] != 'user':
        return False, 'must start with user'
    if msgs[-1]['role'] != 'assistant':
        return False, 'must end with assistant'
    for i, m in enumerate(msgs):
        expected = 'user' if i % 2 == 0 else 'assistant'
        if m['role'] != expected:
            return False, f'role mismatch at index {i}'
        if not str(m['content']).strip():
            return False, f'empty content at index {i}'

    BANNED_OPENERS = [
        'great,', 'great!', 'great.', 'perfect,', 'perfect!', 'perfect.',
        'awesome,', 'awesome!', 'got it', 'noted,', 'noted.',
        'sounds good', 'sounds like a great', 'makes sense', 'thanks!',
    ]
    for m in msgs:
        if m['role'] != 'assistant':
            continue
        first_50 = m['content'].strip()[:50].lower()
        for phrase in BANNED_OPENERS:
            if first_50.startswith(phrase):
                return False, f'banned opener "{phrase}" in assistant message'


    return True, ''


def _log_failure(failures_dir, idx, attempt, err, raw_response):
    """Persist the raw response and the error to disk so failures can be inspected."""
    failures_dir.mkdir(parents=True, exist_ok=True)
    path = failures_dir / f'{idx:05d}_attempt{attempt}.txt'
    body = f'ERROR: {type(err).__name__}: {err}\n\n---RAW RESPONSE---\n{raw_response or "<no response received>"}'
    path.write_text(body, encoding='utf-8')


def generate_dataset(profiles, provider, output_path,
                     system_message=TRAINING_SYSTEM_MESSAGE,
                     max_retries=3, sleep_between=2.5, progress_every=25,
                     max_tokens=8000):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    failures_dir = output_path.parent / 'failures'

    completed = set()
    if output_path.exists():
        with open(output_path) as f:
            for line in f:
                try:
                    completed.add(json.loads(line)['index'])
                except Exception:
                    pass
        print(f'Found {len(completed)} completed records — skipping those.')

    failures = 0
    with open(output_path, 'a') as f:
        for i, profile in enumerate(profiles):
            if i in completed:
                continue

            user_prompt = build_user_prompt(profile)
            success = False

            for attempt in range(max_retries):
                response = None
                try:
                    response = provider.generate(
                        messages=[{'role': 'user', 'content': user_prompt}],
                        temperature=0.95,
                        max_tokens=max_tokens,
                        json_mode=True,
                    )
                    conv = parse_response(response)
                    conv = repair_conversation(conv)
                    ok, reason = validate_conversation(conv)
                    if not ok:
                        raise ValueError(f'validation failed: {reason}')

                    full_messages = [{'role': 'system', 'content': system_message}] + conv['messages']
                    record = {'index': i, 'profile': asdict(profile), 'messages': full_messages}
                    f.write(json.dumps(record, ensure_ascii=False) + '\n')
                    f.flush()
                    success = True
                    break
                except Exception as e:
                    print(f'[{i}] attempt {attempt+1}/{max_retries} failed: {e}')
                    try:
                        _log_failure(failures_dir, i, attempt + 1, e, response)
                    except Exception as log_err:
                        print(f'[{i}] (could not write failure log: {log_err})')
                    time.sleep(2 ** attempt)

            if not success:
                failures += 1
                print(f'[{i}] FAILED after {max_retries} attempts')

            if (i + 1) % progress_every == 0:
                print(f'Progress: {i+1}/{len(profiles)} (failures: {failures})')

            time.sleep(sleep_between)

    print(f'\nDone. Total failures: {failures}/{len(profiles)}')
    if failures:
        print(f'Inspect raw failed responses in: {failures_dir}')

## 5. Test run — 7 conversations (5 normal + 2 forced edge cases)

Always do a small batch before the full run. Look at the output. The 2 forced edge cases test the **scope/refusal** behaviour explicitly — we want to see the coach defer to a professional rather than give medical advice.

**What to look for:**
- Coach asks **one** question per turn
- The coach's first message is NOT formula-praise ("Sounds like a great...")
- Acknowledgement words actually vary (no "Got it / Noted / Perfect" everywhere)
- Edge cases trigger a clear deferral to a qualified professional, not medical advice
- The user's tone matches their `coop_style`

In [ ]:
import random as _r

# 5 random profiles + 2 with forced scenario hints to test refusal behaviour
test_profiles = generate_profiles(n=5, meal_ratio=0.6, seed=42)

_rng = _r.Random(99)
forced_meal = random_meal_profile(_rng)
forced_meal.scenario_hint = 'the user shares concerns about a past struggle with disordered eating'
forced_workout = random_workout_profile(_rng)
forced_workout.scenario_hint = 'the user mentions a recent knee injury when asked about experience'

test_profiles += [forced_meal, forced_workout]

n_with_hint = sum(1 for p in test_profiles if p.scenario_hint)
print(f'Test set: {len(test_profiles)} profiles ({n_with_hint} with scenario hints)')

test_path = f'{DATA_DIR}/test_run.jsonl'
if os.path.exists(test_path):
    os.remove(test_path)

generate_dataset(
    profiles=test_profiles,
    provider=provider,
    output_path=test_path,
    sleep_between=2.5,
    progress_every=1,
)

### Inspect all 7 conversations

Quickly scan each. The last 2 should clearly contain a deferral to a professional.

In [ ]:
with open(test_path) as f:
    records = [json.loads(line) for line in f if line.strip()]

# Summary table first
print(f'{"#":<3} {"type":<8} {"opener":<18} {"coop":<18} scenario')
print('-' * 90)
for r in records:
    p = r['profile']
    hint = (p.get('scenario_hint') or '')[:35]
    print(f"{r['index']:<3} {p['profile_type']:<8} {p['opener_style']:<18} {p['coop_style']:<18} {hint}")

# Then full content of each conversation
for r in records:
    p = r['profile']
    print('\n' + '=' * 70)
    print(f"#{r['index']}  type={p['profile_type']}  opener={p['opener_style']}  coop={p['coop_style']}")
    if p.get('scenario_hint'):
        print(f"SCENARIO: {p['scenario_hint']}")
    print('=' * 70)
    for m in r['messages']:
        if m['role'] == 'system':
            continue
        print(f"\n[{m['role'].upper()}] {m['content']}")

## 6. Full run — 1,500 conversations

If the test looked good, run this. Expect ~65 minutes on Groq's free tier (rate-limited at ~30 RPM).

**Resumable:** if Colab disconnects, re-run this cell — it picks up from where it stopped.

In [9]:
profiles = generate_profiles(n=1500, meal_ratio=0.6, seed=42)
n_meal = sum(1 for p in profiles if p.profile_type == 'meal')
print(f'Generated {len(profiles)} profiles ({n_meal} meal, {len(profiles)-n_meal} workout)')

generate_dataset(
    profiles=profiles,
    provider=provider,
    output_path=RAW_PATH,
    sleep_between=2.5,
)

Generated 1500 profiles (900 meal, 600 workout)
Found 1500 completed records — skipping those.

Done. Total failures: 0/1500


## 7. Dedup with sentence embeddings

Greedy: keep first occurrence, drop anything within cosine 0.92 of an already-kept conversation.

Embeds `first_user_message + last_assistant_message` (the opener + the plan) — the most semantically distinctive parts of each conversation.

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer


def dedup_dataset(input_path, output_path, threshold=0.92,
                  model_name='all-MiniLM-L6-v2', batch_size=64):
    records = []
    with open(input_path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f'Loaded {len(records)} records')

    texts = []
    for r in records:
        msgs = [m for m in r['messages'] if m['role'] != 'system']
        first_user = next((m['content'] for m in msgs if m['role'] == 'user'), '')
        last_asst = next((m['content'] for m in reversed(msgs) if m['role'] == 'assistant'), '')
        texts.append((first_user + ' ' + last_asst)[:3000])

    print(f'Loading embedding model: {model_name}')
    model = SentenceTransformer(model_name)
    embeddings = model.encode(
        texts, show_progress_bar=True, batch_size=batch_size,
        normalize_embeddings=True, convert_to_numpy=True,
    )

    keep_indices = []
    kept_matrix = None
    for i, emb in enumerate(embeddings):
        if kept_matrix is None:
            keep_indices.append(i)
            kept_matrix = emb.reshape(1, -1)
            continue
        sims = kept_matrix @ emb
        if sims.max() < threshold:
            keep_indices.append(i)
            kept_matrix = np.vstack([kept_matrix, emb])

    print(f'Kept {len(keep_indices)} / {len(records)} '
          f'({len(records) - len(keep_indices)} duplicates removed at threshold {threshold})')

    with open(output_path, 'w') as f:
        for i in keep_indices:
            f.write(json.dumps(records[i], ensure_ascii=False) + '\n')

    return len(keep_indices), len(records)


dedup_dataset(RAW_PATH, OUT_PATH, threshold=0.92)
print(f'\nFinal dataset: {OUT_PATH}')

Loaded 1500 records
Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Kept 1407 / 1500 (93 duplicates removed at threshold 0.92)

Final dataset: /content/drive/MyDrive/fitcoach_data/conversations.jsonl


## Done

Your fine-tuning dataset is at `OUT_PATH`. Each line is a JSON object:

```json
{
  "index": 0,
  "profile": {...},
  "messages": [
    {"role": "system", "content": "You are FitCoach..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    ...
  ]
}
```

**Next stage:** QLoRA fine-tuning on LLaMA 3 8B Instruct. New notebook for that one.